# Batch Search On One Loaded LanceDB Slice

This notebook documents the public `import kayak` path for many queries against the same database-backed slice.

Flow:

1. encode text into late-interaction token vectors
2. persist those vectors in LanceDB
3. run `retriever.search_text_batch(...)` as the easy path
4. load one exact slice once with `load_index(...)`
5. reuse that loaded `LateIndex` with `kayak.search_batch(...)`
6. compare local timings for looped search, retriever batch search, and explicit batch search on the reused slice

Install for this notebook with:

```bash
uv add lancedb pyarrow
```

or:

```bash
pixi add --pypi lancedb pyarrow
```

This notebook stays on the public `kayak` surface. It does not assume generic thread-safe concurrent use of one store instance across every adapter. The supported reusable unit here is the loaded `LateIndex`.

In [1]:
import hashlib
import tempfile
import time
from pathlib import Path

import numpy as np

import kayak

BACKEND = kayak.MOJO_EXACT_CPU_BACKEND
BACKEND_INFO = kayak.backend_info(BACKEND)
if not BACKEND_INFO.available:
    raise RuntimeError(BACKEND_INFO.availability_reason)


def token_vectors(text: str) -> np.ndarray:
    rows: list[np.ndarray] = []
    for token in text.lower().split():
        vector = np.zeros(128, dtype=np.float32)
        hot_index = int(hashlib.sha256(token.encode("utf-8")).hexdigest(), 16) % 128
        vector[hot_index] = np.float32(1.0)
        rows.append(vector)
    if not rows:
        return np.zeros((1, 128), dtype=np.float32)
    return np.stack(rows)


def hit_ids(hits: object) -> list[str]:
    return [hit.doc_id for hit in hits]


def timed_avg_ms(fn, *, repeats: int = 50, warmup: int = 5) -> float:
    for _ in range(warmup):
        fn()
    start = time.perf_counter()
    for _ in range(repeats):
        fn()
    return ((time.perf_counter() - start) * 1000.0) / repeats


print("available_backends:", kayak.available_backends())
print("backend_info:", BACKEND_INFO)

available_backends: ('numpy_reference', 'mojo_exact_cpu')
backend_info: BackendInfo(name='mojo_exact_cpu', available=True, requires_mojo=True, query_layouts=('nested', 'flat_dim128'), index_layouts=('packed', 'hybrid_flat_dim128'), availability_reason='Kayak can invoke Mojo via: bash /Users/teilomillet/Code/kayak/scripts/run_mojo_with_pixi_python.sh')


In [2]:
tempdir = tempfile.TemporaryDirectory(prefix="kayak-lancedb-batch-")
store_root = Path(tempdir.name) / "lancedb"

encoder = kayak.open_encoder(
    "callable",
    query_encoder=token_vectors,
    document_encoder=token_vectors,
)
store = kayak.open_store("lancedb", path=store_root, table_name="docs")
retriever = kayak.open_text_retriever(
    encoder=encoder,
    store=store,
    backend=BACKEND,
    default_include_text=True,
)

retriever.upsert_texts(
    ["doc-a", "doc-b", "doc-c", "doc-d", "doc-e"],
    [
        "one environment keeps python mojo and kayak together",
        "lancedb keeps multivector rows durable on disk",
        "kayak can load one exact slice and reuse it for many queries",
        "search batch runs many exact queries on one loaded index",
        "pgvector and qdrant can also stay as durable stores",
    ],
    metadata=[
        {"topic": "installation", "tenant": "acme"},
        {"topic": "storage", "tenant": "acme"},
        {"topic": "search", "tenant": "acme"},
        {"topic": "search", "tenant": "acme"},
        {"topic": "storage", "tenant": "beta"},
    ],
)

query_texts = (
    "python mojo kayak environment",
    "exact batch search on loaded index",
    "durable storage with lancedb",
)

store.stats()

[2026-04-15T16:05:33Z WARN  lance::dataset::write::insert] No existing dataset at /var/folders/j5/d3tnw1ss41g29qttjgv4dhjr0000gn/T/kayak-lancedb-batch-3vnkdrzl/lancedb/docs.lance, it will be created


LateStoreStats(kind='lancedb', document_count=5, total_vector_count=46, vector_dim=128, has_texts=True, has_metadata=True, storage_byte_size=26647)

In [3]:
retriever_hits = retriever.search_text_batch(
    query_texts,
    k=2,
    where={"tenant": "acme"},
    include_text=True,
)

retriever_batch_results = {
    query_texts[index]: [
        {"doc_id": hit.doc_id, "score": round(hit.score, 3)}
        for hit in hits
    ]
    for index, hits in enumerate(retriever_hits)
}
retriever_batch_results

{'python mojo kayak environment': [{'doc_id': 'doc-a', 'score': 4.0},
  {'doc_id': 'doc-c', 'score': 2.0}],
 'exact batch search on loaded index': [{'doc_id': 'doc-d', 'score': 6.0},
  {'doc_id': 'doc-b', 'score': 2.0}],
 'durable storage with lancedb': [{'doc_id': 'doc-b', 'score': 2.0},
  {'doc_id': 'doc-a', 'score': 0.0}]}

In [4]:
index = retriever.load_index(where={"tenant": "acme"}, include_text=True)
encoded_queries = {text: encoder.encode_query(text) for text in query_texts}
batch = kayak.query_batch(
    [encoded_queries[text].as_vector_matrix() for text in query_texts]
)

loop_hits = [
    kayak.search(encoded_queries[text], index, k=2, backend=BACKEND)
    for text in query_texts
]
explicit_batch_hits = kayak.search_batch(
    batch,
    index,
    k=2,
    backend=BACKEND,
)

loop_ids = [hit_ids(hits) for hits in loop_hits]
retriever_batch_ids = [hit_ids(hits) for hits in retriever_hits]
explicit_batch_ids = [hit_ids(hits) for hits in explicit_batch_hits]

assert loop_ids == retriever_batch_ids == explicit_batch_ids

{
    "loaded_index_doc_ids": index.doc_ids,
    "loaded_index_vector_counts": index.vector_counts,
    "batch_query_vector_counts": batch.vector_counts,
    "top_ids_by_query": {
        query_texts[index]: explicit_batch_ids[index]
        for index in range(len(query_texts))
    },
}

{'loaded_index_doc_ids': ('doc-a', 'doc-b', 'doc-c', 'doc-d'),
 'loaded_index_vector_counts': (8, 7, 12, 10),
 'batch_query_vector_counts': (4, 6, 4),
 'top_ids_by_query': {'python mojo kayak environment': ['doc-a', 'doc-c'],
  'exact batch search on loaded index': ['doc-d', 'doc-b'],
  'durable storage with lancedb': ['doc-b', 'doc-a']}}

In [5]:
def loop_retriever_search() -> object:
    return [
        retriever.search_text(
            text,
            k=2,
            where={"tenant": "acme"},
            include_text=True,
        )
        for text in query_texts
    ]


def retriever_batch_search() -> object:
    return retriever.search_text_batch(
        query_texts,
        k=2,
        where={"tenant": "acme"},
        include_text=True,
    )


def explicit_loaded_slice_batch_search() -> object:
    return kayak.search_batch(batch, index, k=2, backend=BACKEND)


loop_ms = timed_avg_ms(loop_retriever_search)
retriever_batch_ms = timed_avg_ms(retriever_batch_search)
explicit_batch_ms = timed_avg_ms(explicit_loaded_slice_batch_search)

{
    "retriever_search_text_loop_ms": round(loop_ms, 3),
    "retriever_search_text_batch_ms": round(retriever_batch_ms, 3),
    "explicit_loaded_slice_search_batch_ms": round(explicit_batch_ms, 3),
    "retriever_batch_speedup_vs_loop": round(loop_ms / retriever_batch_ms, 3),
    "explicit_loaded_slice_speedup_vs_loop": round(loop_ms / explicit_batch_ms, 3),
}

{'retriever_search_text_loop_ms': 9.851,
 'retriever_search_text_batch_ms': 9.578,
 'explicit_loaded_slice_search_batch_ms': 0.064,
 'retriever_batch_speedup_vs_loop': 1.029,
 'explicit_loaded_slice_speedup_vs_loop': 154.42}

In [6]:
retriever.close()
tempdir.cleanup()
print("cleaned up temporary LanceDB state")

cleaned up temporary LanceDB state
